# Florence-2: Detection & Grounding on Satellite Imagery

**Goal:** Test whether Florence-2, running **locally** on Apple Silicon (MPS), can detect and localize hydraulic features in **satellite imagery**.

**Context:** In attempt1, Florence-2 struggled with DEM hillshade — detecting only 1 object via `<OD>` and producing whole-image bboxes. Satellite imagery should be far more in-distribution for Florence-2, which was trained on natural photos.

**Roadmap reference:** Stage 6.3 — Florence-2 Detection on Satellite

## Tasks
1. Run `<OD>` (object detection) on satellite RGB
2. Run `<CAPTION_TO_PHRASE_GROUNDING>` with hydraulic feature phrases
3. Run `<DENSE_REGION_CAPTION>` for open-ended detection
4. Compare detection quality to attempt1 DEM results
5. Save timestamped outputs to `data/output/model-outputs/attempt2/florence2/`

In [ ]:
import os
import json
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import rasterio
from PIL import Image

# Paths
DEM_PATH = Path("../../data/input/1m elevation.tif")
SATELLITE_PATH = Path("../../data/output/cimarron_satellite.png")
HILLSHADE_PATH = Path("../../data/output/cimarron_hillshade.png")
OUTPUT_DIR = Path("../../data/output/model-outputs/attempt2/florence2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Timestamp for this run
RUN_TS = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

print(f"DEM: {DEM_PATH}")
print(f"Satellite image: {SATELLITE_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Run timestamp: {RUN_TS}")

## 1.1 Load Florence-2 Model

Uses `florence-community/Florence-2-large` — native transformers checkpoint. First run downloads ~3.1 GB.

In [ ]:
import torch
from transformers import AutoProcessor, Florence2ForConditionalGeneration

FLORENCE_MODEL_ID = "florence-community/Florence-2-large"
FLORENCE_AVAILABLE = False

try:
    print(f"Loading {FLORENCE_MODEL_ID}...")
    print(f"Device: MPS (Apple Silicon)")
    print()

    processor = AutoProcessor.from_pretrained(FLORENCE_MODEL_ID)
    model = Florence2ForConditionalGeneration.from_pretrained(
        FLORENCE_MODEL_ID,
        torch_dtype=torch.float16,
    ).to("mps")

    param_count = sum(p.numel() for p in model.parameters())
    print(f"Model loaded: {FLORENCE_MODEL_ID}")
    print(f"Parameters: {param_count / 1e6:.0f}M")
    print(f"Device: {next(model.parameters()).device}")
    FLORENCE_AVAILABLE = True

except Exception as e:
    print(f"Failed to load Florence-2.")
    print(f"Error: {e}")
    print()
    print("Possible fixes:")
    print("  - pip install transformers torch timm einops")
    print("  - Ensure you have ~4GB free RAM")
    print()
    print("The rest of this notebook will be skipped.")

## 1.2 Load Images & DEM Metadata

In [ ]:
# Load DEM metadata for geo-coordinate conversion
with rasterio.open(DEM_PATH) as src:
    dem_transform = src.transform
    dem_bounds = src.bounds
    dem_crs = src.crs

print(f"CRS: {dem_crs}")
print(f"Bounds: {dem_bounds}")

# Load satellite image
if not SATELLITE_PATH.exists():
    raise FileNotFoundError(
        f"Satellite image not found at {SATELLITE_PATH}.\n"
        "Run the satellite acquisition notebook (Stage 5) first."
    )

img_satellite = Image.open(SATELLITE_PATH)
print(f"Satellite image: {img_satellite.size} ({img_satellite.mode})")

## 1.3 Inference Utilities

In [ ]:
def florence_run(image, task, text_input=""):
    """Run Florence-2 inference with a task token."""
    prompt = task + text_input
    inputs = processor(text=prompt, images=image, return_tensors="pt").to("mps")

    with torch.no_grad():
        generated = model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=1024,
            num_beams=3,
        )

    decoded = processor.batch_decode(generated, skip_special_tokens=False)[0]
    parsed = processor.post_process_generation(
        decoded, task=task, image_size=(image.width, image.height)
    )
    return parsed


def extract_bboxes(result, task):
    """Extract bboxes and labels from Florence-2 result dict."""
    detections = []
    task_result = result.get(task, {})
    bboxes = task_result.get("bboxes", [])
    labels = task_result.get("labels", [])

    for i, bbox in enumerate(bboxes):
        label = labels[i] if i < len(labels) else f"detection_{i}"
        detections.append({"coords": [float(c) for c in bbox], "label": label})

    return detections


def plot_bboxes_on_image(img, bboxes, labels, title, save_path=None, figsize=(14, 8)):
    """Overlay bounding boxes and labels on an image."""
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.imshow(img)
    ax.set_title(title, fontsize=14)

    colors = plt.cm.Set1(np.linspace(0, 1, max(len(bboxes), 1)))

    for i, (bbox, label) in enumerate(zip(bboxes, labels)):
        x1, y1, x2, y2 = bbox
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=colors[i % len(colors)], facecolor="none",
        )
        ax.add_patch(rect)
        ax.text(x1, y1 - 5, label, fontsize=8, color=colors[i % len(colors)],
                bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7))

    ax.set_xlabel(f"{len(bboxes)} detections" if bboxes else "No detections")
    ax.set_xticks([])
    ax.set_yticks([])
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()
    return fig


all_results = {}
print("Utilities ready.")

## 2.1 Object Detection — Satellite

Run open-vocabulary object detection (`<OD>`) on the satellite image.

In [ ]:
if not FLORENCE_AVAILABLE:
    print("Skipping — Florence-2 is not available.")
else:
    task = "<OD>"

    print(f"Running {task} on satellite ({img_satellite.size})...")
    t0 = time.time()
    result = florence_run(img_satellite, task)
    elapsed = time.time() - t0
    print(f"Inference time: {elapsed:.1f}s")

    all_results["od_satellite"] = {"task": task, "result": result, "time": elapsed}

    detections = extract_bboxes(result, task)
    print(f"\nDetections: {len(detections)}")
    for det in detections:
        x1, y1, x2, y2 = det["coords"]
        print(f"  ({x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f}) — {det['label']}")

    bboxes = [d["coords"] for d in detections]
    labels = [d["label"] for d in detections]
    save_path = OUTPUT_DIR / f"{RUN_TS}_satellite_od.png"
    plot_bboxes_on_image(img_satellite, bboxes, labels,
                         f"Florence-2 Object Detection (satellite) — {len(detections)} detections",
                         save_path=save_path)

## 2.2 Phrase Grounding — Satellite

Use `<CAPTION_TO_PHRASE_GROUNDING>` with targeted hydraulic feature phrases.

In [ ]:
if not FLORENCE_AVAILABLE:
    print("Skipping — Florence-2 is not available.")
else:
    task = "<CAPTION_TO_PHRASE_GROUNDING>"

    phrases = {
        "hydraulic": "river channel. floodplain. sand bar. point bar. bridge.",
        "infrastructure": "road. bridge. building. fence. embankment.",
        "vegetation": "forest. grassland. agricultural field. bare soil. riparian zone.",
    }

    for phrase_key, phrase in phrases.items():
        test_name = f"phrase_satellite_{phrase_key}"
        print(f"\n{'='*80}")
        print(f"TEST: {test_name}")
        print(f"Phrase: {phrase}")
        print(f"{'='*80}")

        t0 = time.time()
        result = florence_run(img_satellite, task, text_input=phrase)
        elapsed = time.time() - t0
        print(f"Inference time: {elapsed:.1f}s")

        all_results[test_name] = {
            "task": task, "phrase": phrase, "result": result, "time": elapsed,
        }

        detections = extract_bboxes(result, task)
        print(f"\nDetections: {len(detections)}")
        for det in detections:
            x1, y1, x2, y2 = det["coords"]
            print(f"  ({x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f}) — {det['label']}")

        bboxes = [d["coords"] for d in detections]
        labels = [d["label"] for d in detections]
        save_path = OUTPUT_DIR / f"{RUN_TS}_satellite_phrase_{phrase_key}.png"
        plot_bboxes_on_image(img_satellite, bboxes, labels,
                             f"Florence-2 Phrase Grounding: {phrase_key} (satellite)",
                             save_path=save_path)

## 2.3 Dense Region Captioning — Satellite

In [ ]:
if not FLORENCE_AVAILABLE:
    print("Skipping — Florence-2 is not available.")
else:
    task = "<DENSE_REGION_CAPTION>"

    print(f"Running {task} on satellite ({img_satellite.size})...")
    t0 = time.time()
    result = florence_run(img_satellite, task)
    elapsed = time.time() - t0
    print(f"Inference time: {elapsed:.1f}s")

    all_results["dense_satellite"] = {"task": task, "result": result, "time": elapsed}

    detections = extract_bboxes(result, task)
    print(f"\nDetections: {len(detections)}")
    for det in detections:
        x1, y1, x2, y2 = det["coords"]
        print(f"  ({x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f}) — {det['label']}")

    bboxes = [d["coords"] for d in detections]
    labels = [d["label"] for d in detections]
    save_path = OUTPUT_DIR / f"{RUN_TS}_satellite_dense_caption.png"
    plot_bboxes_on_image(img_satellite, bboxes, labels,
                         f"Florence-2 Dense Captioning (satellite) — {len(detections)} detections",
                         save_path=save_path)

## 3. Results Summary & Save

In [ ]:
if not FLORENCE_AVAILABLE:
    print("Florence-2 was not available — no results to summarize.")
else:
    # Results table
    print("FLORENCE-2 DETECTION RESULTS (SATELLITE)")
    print(f"{'Test':<30} {'Task':<35} {'Detections':>10} {'Time':>8}")
    print("-" * 90)
    for key, result in all_results.items():
        task = result["task"]
        detections = extract_bboxes(result["result"], task)
        n = len(detections)
        t = result["time"]
        print(f"{key:<30} {task:<35} {n:>10} {t:>7.1f}s")

    print(f"\nModel: {FLORENCE_MODEL_ID}")
    print(f"Total tests: {len(all_results)}")

    # Save results to markdown
    md_lines = [
        f"# Florence-2 — Satellite Detection Test Results\n",
        f"\n**Date:** {RUN_TS}\n",
        f"\n**Model:** `{FLORENCE_MODEL_ID}` (local, MPS)\n",
        f"\n**Input:** Satellite (NAIP RGB, 0.6m/pixel)\n",
        f"\n**DEM:** Cimarron River, 1m resolution, {dem_crs}\n",
        f"\n**Notebook:** `notebooks/attempt2/04_florence2_satellite.ipynb`\n",
        f"\n**Attempt:** 2 (satellite imagery)\n",
        "\n---\n",
    ]

    for key, result in all_results.items():
        task = result["task"]
        detections = extract_bboxes(result["result"], task)

        md_lines.append(f"\n## {key.replace('_', ' ').title()}\n")
        md_lines.append(f"\n**Task:** `{task}`\n")
        if "phrase" in result:
            md_lines.append(f"\n**Phrase:** {result['phrase']}\n")
        md_lines.append(f"\n**Inference time:** {result['time']:.1f}s\n")
        md_lines.append(f"\n**Detections:** {len(detections)}\n")

        if detections:
            md_lines.append("\n**Bounding boxes (pixel coords):**\n\n")
            for det in detections:
                x1, y1, x2, y2 = det["coords"]
                md_lines.append(f"- ({x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f}) — {det['label']}\n")

            # Convert bbox centers to geo coords
            geo_points = []
            for det in detections:
                x1, y1, x2, y2 = det["coords"]
                cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
                geo_points.append((cx, cy, det["label"]))
            if geo_points:
                md_lines.append(f"\n**Geo-coordinates ({dem_crs}, bbox centers):**\n\n")
                for x_px, y_px, label in geo_points:
                    e, n = rasterio.transform.xy(dem_transform, int(y_px), int(x_px))
                    md_lines.append(f"- ({e:.1f}, {n:.1f}) — {label}\n")

        md_lines.append(f"\n**Raw result:**\n\n```json\n{json.dumps(result['result'], indent=2, default=str)[:2000]}\n```\n")
        md_lines.append("\n---\n")

    # Summary table
    md_lines.append("\n## Summary\n\n")
    md_lines.append("| Test | Task | Detections | Time |\n")
    md_lines.append("|------|------|------------|------|\n")
    for key, result in all_results.items():
        task = result["task"]
        detections = extract_bboxes(result["result"], task)
        n = len(detections)
        t = result["time"]
        md_lines.append(f"| {key} | `{task}` | {n} | {t:.1f}s |\n")

    output_path = OUTPUT_DIR / f"{RUN_TS}_florence2_satellite_results.md"
    with open(output_path, "w") as f:
        f.writelines(md_lines)

    print(f"\nResults saved to: {output_path}")